## Regression

The regression task is to predict the average daily rate of a hotel booking. The target variable, `adr`, is continuous and represents the average room revenue per occupied room per day.

In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import GridSearchCV
from sklearn.feature_selection import SelectKBest, f_regression

import numpy as np

from sklearn import set_config

set_config(display="text")

In [ ]:
df = pd.read_csv("data/hotel_bookings_clean.csv")
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,0,C,C,3,No Deposit,0,Transient,0.0,0,0
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,0,C,C,4,No Deposit,0,Transient,0.0,0,0
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,0,A,C,0,No Deposit,0,Transient,75.0,0,0
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,0,A,A,0,No Deposit,0,Transient,75.0,0,0
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,0,A,A,0,No Deposit,0,Transient,98.0,0,1


In [3]:
df.nunique()

hotel                                2
is_canceled                          2
lead_time                          479
arrival_date_year                    3
arrival_date_month                  12
arrival_date_week_number            53
arrival_date_day_of_month           31
stays_in_weekend_nights             17
stays_in_week_nights                33
adults                              14
children                             5
babies                               5
meal                                 5
country                            178
market_segment                       7
distribution_channel                 5
is_repeated_guest                    2
previous_cancellations              15
previous_bookings_not_canceled      73
reserved_room_type                   9
assigned_room_type                  11
booking_changes                     19
deposit_type                         3
days_in_waiting_list               127
customer_type                        4
adr                      

In [4]:
# Change non numeric columns to numeric
df = pd.get_dummies(df, drop_first=True)
df

,is_canceled,lead_time,arrival_date_year,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,...,assigned_room_type_G,assigned_room_type_H,assigned_room_type_I,assigned_room_type_K,assigned_room_type_L,deposit_type_Non Refund,deposit_type_Refundable,customer_type_Group,customer_type_Transient,customer_type_Transient-Party
0,0,342,2015,27,1,0,0,2,0.0,0,...,False,False,False,False,False,False,False,False,True,False
1,0,737,2015,27,1,0,0,2,0.0,0,...,False,False,False,False,False,False,False,False,True,False
2,0,7,2015,27,1,0,1,1,0.0,0,...,False,False,False,False,False,False,False,False,True,False
3,0,13,2015,27,1,0,1,1,0.0,0,...,False,False,False,False,False,False,False,False,True,False
4,0,14,2015,27,1,0,2,2,0.0,0,...,False,False,False,False,False,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86934,0,23,2017,35,30,2,5,2,0.0,0,...,False,False,False,False,False,False,False,False,True,False
86935,0,102,2017,35,31,2,5,3,0.0,0,...,False,False,False,False,False,False,False,False,True,False
86936,0,34,2017,35,31,2,5,2,0.0,0,...,False,False,False,False,False,False,False,False,True,False
86937,0,109,2017,35,31,2,5,2,0.0,0,...,False,False,False,False,False,False,False,False,True,False


In [ ]:
# Create X and Y
X = df.drop(columns="adr")

y = df["adr"]

In [6]:
# Split into sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
# Find numeric columns
numeric_cols = X_train.select_dtypes(include="number").columns

In [8]:
# Scale numeric columns only
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

In [9]:
# Models
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.1, max_iter=10000),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Goes through each model in the dictionary
for name, model in models.items():
    # Cross Validation
    scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring="r2")

    print(f"{name}: {scores.mean():.4f}")

Linear Regression: 0.5855
Ridge: 0.5857
Lasso: 0.5768
Random Forest: 0.7769
Gradient Boosting: 0.7036


## Optimize

In [10]:
# Hyperparameter Tuning
parameters = {
    "Random Forest": {
        "n_estimators": [100, 200],
        "max_depth": [10, None],
        "min_samples_leaf": [1, 5]
    },
    "Gradient Boosting": {
        "n_estimators": [100, 200],
        "learning_rate": [0.05, 0.1],
        "max_depth": [2, 3]
    }
}

for name in ["Random Forest", "Gradient Boosting"]:
    # Create grid search to test all combinations of parameters
    grid_search = GridSearchCV(models[name], param_grid=parameters[name], cv=cv, scoring="r2")

    # Train each parameter
    grid_search.fit(X_train_scaled, y_train)

    # Print best parameter
    print(name, grid_search.best_params_)

Random Forest {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 200}
Gradient Boosting {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200}


In [11]:
# Rerun the models
rf = RandomForestRegressor(max_depth=None, min_samples_leaf=1, n_estimators=200, random_state=42)
gb = GradientBoostingRegressor(learning_rate=0.1, max_depth=3, n_estimators=200, random_state=42)

# Cross Validation
scores_rf = cross_val_score(rf, X_train_scaled, y_train, cv=cv, scoring="r2")
scores_gb = cross_val_score(gb, X_train_scaled, y_train, cv=cv, scoring="r2")

print(f"Random Forest: {scores_rf.mean():.4f}")
print(f"Gradient Boosting: {scores_gb.mean():.4f}")

Random Forest: 0.7782
Gradient Boosting: 0.7347


In [13]:
# ANOVA
for i in [10, 25, 50, 75, 100, 125, 150, 175, 200]:
    selector = SelectKBest(score_func=f_regression, k=i)

    # Learn which columns to keep from training data only
    selector.fit(X_train_scaled, y_train)

    # Keep those same columns in both sets
    X_train_selected = selector.transform(X_train_scaled)
    X_test_selected = selector.transform(X_test_scaled)

    # Cross Validation
    scores_rf = cross_val_score(rf, X_train_selected, y_train, cv=cv, scoring="r2")
    scores_gb = cross_val_score(gb, X_train_selected, y_train, cv=cv, scoring="r2")

    print(f"Random Forest: {scores_rf.mean():.4f}")
    print(f"Gradient Boosting: {scores_gb.mean():.4f}")

Random Forest: 0.4060
Gradient Boosting: 0.4020
Random Forest: 0.5293
Gradient Boosting: 0.5349
Random Forest: 0.6972
Gradient Boosting: 0.7108
Random Forest: 0.7769
Gradient Boosting: 0.7344
Random Forest: 0.7766
Gradient Boosting: 0.7334
Random Forest: 0.7778
Gradient Boosting: 0.7338
Random Forest: 0.7762
Gradient Boosting: 0.7330
Random Forest: 0.7773
Gradient Boosting: 0.7313
Random Forest: 0.7767
Gradient Boosting: 0.7313


In [17]:
# Retrain model with ANOVA
selector = SelectKBest(score_func=f_regression, k=75)

# Learn which columns to keep from training data only
selector.fit(X_train_scaled, y_train)

# Keep those same columns in both sets
X_train_selected = selector.transform(X_train_scaled)
X_test_selected = selector.transform(X_test_scaled)

# Train model
rf.fit(X_train_selected, y_train)

# Predict
y_pred_rf = rf.predict(X_test_selected)

## Evaluate

In [18]:
# Errors
print(f"R²:   {r2_score(y_test, y_pred_rf):.4f}")
print(f"MAE:  {mean_absolute_error(y_test, y_pred_rf):.2f}")
print(f"RMSE: {root_mean_squared_error(y_test, y_pred_rf):.2f}")

R²:   0.8474
MAE:  11.32
RMSE: 20.45


Correctly explains 84.74% of the variation in room rates (R2).

Predictions are off by $11.32 on average (MAE)

RMSE of $20.45 shows model is sensitive to large numbers since much different from absolute.